# Sentiment Model Demo

This notebook loads our fine tuned Cardiff RoBERTa sentiment classifier from Hugging Face and runs it against a sample of Twitch chat messages. It pulls a non-random contiguous window of 25 messages from `test.txt` and displays the predicted sentiment label and confidence score for each message. If you want to increase the window size, adjust `WIN`. 

In [1]:
from pathlib import Path
import html
import os
import re
from collections import Counter

from dotenv import load_dotenv
from IPython.display import HTML, display

from display_bert import SentModel

C:\Users\jason\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CHAT_LINE_RE = re.compile(r"^\[(?P<timestamp>[^\]]+)\]\s+(?P<username>[^:]+):\s*(?P<text>.*)$")


def load_env(start_dir: Path) -> None:
    for base in [start_dir, *start_dir.parents]:
        env_path = base / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            return
    load_dotenv()


def pick_input(app_dir: Path) -> Path:
    candidates = [app_dir / "test.txt", app_dir / "text.txt"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Expected `test.txt` (or `text.txt`) in the notebook directory.")


def load_msgs(chat_file: Path) -> list[dict]:
    msgs = []
    with open(chat_file, "r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if not line.strip():
                continue
            match = CHAT_LINE_RE.match(line)
            if not match:
                continue
            msgs.append(
                {
                    "timestamp": match.group("timestamp"),
                    "username": match.group("username"),
                    "text": match.group("text"),
                }
            )
    return msgs


In [3]:
APP_DIR = Path.cwd()
load_env(APP_DIR)
INPUT_FILE = pick_input(APP_DIR)

WIN = 25
START = 0

model_id = os.getenv("MODEL_ID", "").strip()
if not model_id:
    raise RuntimeError("MODEL_ID is required in .env for Hugging Face model loading.")

clf = SentModel(
    model_src=model_id,
    max_len=int(os.getenv("BERT_MAX_LENGTH", "128")),
    use_emote_tags=False,
    emote_lex="",
    min_conf=float(os.getenv("BERT_MIN_CONFIDENCE", "0.55")),
    min_gap=float(os.getenv("BERT_MIN_MARGIN", "0.10")),
    use_prior=os.getenv("BERT_USE_PRIOR_CORRECTION", "0").strip().lower() in {"1", "true", "yes", "on"},
    target_priors=os.getenv("BERT_TARGET_PRIORS", ""),
    train_priors=os.getenv("BERT_TRAIN_PRIORS", ""),
    norm_twitter=False,
    rev=os.getenv("MODEL_REVISION", "").strip() or None,
    token=os.getenv("HF_TOKEN", "").strip() or os.getenv("HUGGINGFACE_HUB_TOKEN", "").strip() or None,
    local_only=False,
)

msgs = load_msgs(INPUT_FILE)
window = msgs[START: START + WIN]
if len(window) < WIN:
    raise ValueError(f"Only found {len(window)} parseable lines in the selected window; need {WIN}.")

rows = []
win_counts = Counter()

for msg in window:
    sentiment, confidence = clf.predict(msg["text"])
    win_counts[sentiment] += 1
    rows.append(
        {
            "timestamp": msg["timestamp"],
            "username": msg["username"],
            "text": msg["text"],
            "sentiment": sentiment,
            "confidence": confidence,
        }
    )

print(f"Model source: {model_id}")
print(f"Input file: {INPUT_FILE.name}")
print(f"Window: msgs {START}..{START + len(window) - 1} ({len(window)} total)")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1688.59it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Model source: JDMates/TwitchRoBERTaSentiment
Input file: test.txt
Window: msgs 0..24 (25 total)


In [4]:
def show_chat(items: list[dict]) -> None:
    css = """
    <style>
      .chat-window { background:#111622; border:1px solid #20263a; border-radius:12px; padding:10px 12px; }
      .msg-row { padding:5px 0; line-height:1.35; word-break:break-word; color:#e9edf7; font-family:'Segoe UI', Tahoma, sans-serif; }
      .msg-time { color:#7e89a8; margin-right:8px; }
      .msg-sent { display:inline-block; min-width:118px; text-align:center; margin-right:6px; border-radius:999px; font-size:12px; line-height:1.2; white-space:nowrap; padding:2px 8px; border:1px solid transparent; }
      .msg-sent.positive { color:#22c55e; border-color:rgba(34,197,94,0.5); }
      .msg-sent.neutral { color:#f59e0b; border-color:rgba(245,158,11,0.5); }
      .msg-sent.negative { color:#ef4444; border-color:rgba(239,68,68,0.5); }
      .msg-user { font-weight:700; margin-right:6px; }
      .msg-text { color:#e9edf7; }
    </style>
    """

    rendered_rows = []
    for item in items:
        sent_class = str(item["sentiment"]).lower()
        if sent_class not in {"positive", "neutral", "negative"}:
            sent_class = "neutral"
        conf_pct = int(round(float(item["confidence"]) * 100))
        rendered_rows.append(
            """
            <div class='msg-row'>
              <span class='msg-time'>{timestamp}</span>
              <span class='msg-sent {sent_class}'>{sentiment} | {conf_pct}%</span>
              <span class='msg-user'>{username}:</span>
              <span class='msg-text'>{text}</span>
            </div>
            """.format(
                timestamp=html.escape(str(item["timestamp"])),
                sent_class=sent_class,
                sentiment=html.escape(str(item["sentiment"])),
                conf_pct=conf_pct,
                username=html.escape(str(item["username"])),
                text=html.escape(str(item["text"])),
            )
        )

    display(HTML(css + "<div class='chat-window'>" + "".join(rendered_rows) + "</div>"))


show_chat(rows)

In [5]:
def show_win_pct(counts: Counter, total: int) -> None:
    pct_pos = (counts["Positive"] / total * 100.0) if total else 0.0
    pct_neu = (counts["Neutral"] / total * 100.0) if total else 0.0
    pct_neg = (counts["Negative"] / total * 100.0) if total else 0.0

    summary_html = f"""
    <style>
      .sentiment-pills {{ display:flex; gap:8px; flex-wrap:wrap; margin-top:10px; font-family:'Segoe UI', Tahoma, sans-serif; }}
      .pill {{ font-size:12px; border-radius:999px; border:1px solid #20263a; padding:4px 10px; white-space:nowrap; }}
      .pill.positive {{ color:#22c55e; }}
      .pill.neutral {{ color:#f59e0b; }}
      .pill.negative {{ color:#ef4444; }}
    </style>
    <div class='sentiment-pills'>
      <span class='pill positive'>Positive {pct_pos:.1f}% ({counts['Positive']}/{total})</span>
      <span class='pill neutral'>Neutral {pct_neu:.1f}% ({counts['Neutral']}/{total})</span>
      <span class='pill negative'>Negative {pct_neg:.1f}% ({counts['Negative']}/{total})</span>
    </div>
    """
    display(HTML(summary_html))


show_win_pct(win_counts, len(rows))